<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 2 — Fitting Models: What Sets the Tide?</div>
</div>

#### Fit a tidal model to the Lancaster Quay record from Week 1: test the procedure on generated data, fit with the period fixed and then free, find out what the model is missing, and judge the result.
***

## Where this fits

In Week 1 you looked at a week of water levels from Lancaster Quay. The peaks were about 12.3 hours apart, close to the 12.42-hour M2 lunar tide, and the distribution was strongly skewed. Lecture 3 then showed that fitting a model means **minimising χ²**, that a model can be *linear in its parameters* even when it describes an oscillation, and that a fit is only finished once you have judged it: residuals, χ²/ν, uncertainties and correlations.

Today you fit a tidal model to that record, following the lecture's workflow:

1. test the fitting procedure on generated data with known answers;
2. fit the real record with the tidal period fixed (a linear fit);
3. let the period float (a nonlinear fit) and see what it finds;
4. work out what is missing from the model, and judge the final fit.

The figure you will make repeatedly (data and model on top, residuals underneath) is covered in detail in `week2_code_and_plots.ipynb`.

## <span style="color:#B5121B">0. Load the data (provided)</span>

In [ ]:
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week2/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

tide = pd.read_csv(data_path("lancaster_quay_tidal.csv"))
tide["time"] = pd.to_datetime(tide["time"])

# Time in hours since the first reading, and the level in metres
t = (tide["time"] - tide["time"].iloc[0]).dt.total_seconds().to_numpy() / 3600
level = tide["level_m"].to_numpy()
print(f"{len(t)} readings over {t[-1]:.1f} hours")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t, level, ".", ms=3, color="#555656")
ax.set_xlabel("Time since first reading (h)")
ax.set_ylabel("Water level (m)")
ax.set_title("Lancaster Quay, 24 Sep - 2 Oct 2026")
plt.show()

## <span style="color:#B5121B">1. The model</span>

The simplest tide is one sinusoid of period $P$:

$$h(t) = c + a\cos\omega t + b\sin\omega t, \qquad \omega = \frac{2\pi}{P}.$$

The amplitude is $A = \sqrt{a^2 + b^2}$. Written this way, with $P$ known, the model is **linear in its parameters** $(c, a, b)$, even though it oscillates (Lecture 3, slide 18). Each extra frequency adds one more cosine and sine column, and the model stays linear as long as every period is fixed.

For a linear model, build the **design matrix**: one column per parameter, each evaluated at every time, divided by $\sigma$ (slide 17):

$$A = \frac{1}{\sigma}\begin{pmatrix} 1 & \cos\omega_1 t_1 & \sin\omega_1 t_1 & \cos\omega_2 t_1 & \dots \\ \vdots & & & & \end{pmatrix}, \qquad \mathbf{b} = \mathbf{y}/\sigma,$$

then solve $A\boldsymbol{\theta} \approx \mathbf{b}$ with `np.linalg.lstsq`. The covariance of the parameters is $C = (A^TA)^{-1}$.

Write three functions:

* `chi2(y, y_model, sigma)`: $\chi^2 = \sum_i \left[(y_i - f_i)/\sigma_i\right]^2$ (Lecture 3, slide 15). You will use it for every fit today. (Store results in a variable such as `chi2_val`, not `chi2`, or you will overwrite the function.)
* `design_matrix(t, periods)`: a column of ones, then a cosine and a sine column for each period in the list;
* `linear_fit(t, y, sigma, periods)`: returns the parameters, their covariance matrix, χ² (from your `chi2`) and the degrees of freedom ν = N − p.

In [ ]:
def chi2(y, y_model, sigma):
    """Chi-squared of data y against model values y_model, with uncertainties sigma."""
    # TODO: one line, using np.sum
    raise NotImplementedError

def design_matrix(t, periods):
    """Columns: 1, then cos(2 pi t / P) and sin(2 pi t / P) for each period P."""
    # TODO: start with a list holding np.ones_like(t), add two columns per period,
    #       then return np.column_stack(columns)
    raise NotImplementedError

def linear_fit(t, y, sigma, periods):
    """Weighted linear least squares. Returns (theta, cov, chi2_value, nu)."""
    # TODO: A = design_matrix(t, periods) / sigma; b = y / sigma
    # TODO: theta = np.linalg.lstsq(A, b, rcond=None)[0]
    # TODO: cov = np.linalg.inv(A.T @ A)
    # TODO: chi2_value = chi2(y, model values, sigma); nu = len(y) - len(theta)
    raise NotImplementedError

## <span style="color:#B5121B">2. Test on generated data first</span>

Before fitting real data, check that the procedure recovers known answers (Lecture 3, Part D). The cell below makes a fake tide at the same times as the real one: $c = 2.8$ m, $a = 0.6$ m, $b = 0.8$ m, $P = 12.4206$ h (M2), plus Gaussian noise with $\sigma = 0.05$ m.

In [ ]:
M2 = 12.4206            # M2 lunar semi-diurnal period, hours
true = {"c": 2.8, "a": 0.6, "b": 0.8}
sigma_fake = 0.05       # m

rng = np.random.default_rng(1)
def make_fake_tide():
    return (true["c"] + true["a"] * np.cos(2 * np.pi * t / M2)
            + true["b"] * np.sin(2 * np.pi * t / M2) + rng.normal(0, sigma_fake, t.size))

fake = make_fake_tide()

Fit `fake` with `linear_fit` (period M2). Print each parameter with its uncertainty next to the true value, and χ²/ν. Are the true values within about 1σ? Is χ²/ν close to 1, within $\pm\sqrt{2/\nu}$?

In [ ]:
try:
    # TODO: theta, cov, chi2_val, nu = linear_fit(t, fake, sigma_fake, [M2])
    # TODO: print true value, fitted value and uncertainty for c, a, b; print chi2/nu
    raise NotImplementedError
except NotImplementedError:
    print("Not implemented yet: complete the functions above, then re-run this cell.")

### <span style="color:#FD9029">Optional: is one success luck?</span>

One success could be luck. Repeat the fake experiment 500 times and histogram the **pull** of $a$, $z = (\hat a - a_\text{true})/\sigma_{\hat a}$. If the fit is unbiased and its error bars are honest, the pulls have mean 0 and standard deviation 1 (lecture slide 29).

In [ ]:
try:
    # TODO: loop 500 times: fake = make_fake_tide(); fit; store (theta[1] - true["a"]) / sqrt(cov[1, 1])
    # TODO: print mean and std of the pulls; plot a histogram with a unit Gaussian on top
    raise NotImplementedError
except NotImplementedError:
    print("Not implemented yet: complete the functions above, then re-run this cell.")

## <span style="color:#B5121B">3. Fit the real record with the period fixed</span>

We need $\sigma$ for the real readings. On the low-water plateau (Week 1) the water barely moves, so the scatter there estimates the gauge noise:

In [ ]:
plateau = level < 2.24
sigma_gauge = level[plateau].std()
print(f"{plateau.sum()} plateau readings, scatter {sigma_gauge * 1000:.1f} mm")
sigma_real = 0.004   # m: rounded up, and used below

Fit three models, all linear with fixed periods:

1. M2 only: `[M2]`;
2. M2 and its first two harmonics: `[M2, M2/2, M2/3]` (the tide here isn't symmetric, Week 1);
3. as (2) plus `M2/4`.

For each, print the number of parameters, the rms of the residuals and χ²/ν. Then plot the data with model (2) on top and the residuals in units of σ underneath.

In [ ]:
models = {"M2": [M2],
          "M2 + 2 harmonics": [M2, M2 / 2, M2 / 3],
          "M2 + 3 harmonics": [M2, M2 / 2, M2 / 3, M2 / 4]}
try:
    # TODO: for each model: fit with linear_fit(t, level, sigma_real, periods);
    #       model values = design_matrix(t, periods) @ theta; print p, rms residual, chi2/nu
    # TODO: fit-and-residuals figure for "M2 + 2 harmonics"
    raise NotImplementedError
except NotImplementedError:
    print("Not implemented yet: complete the functions above, then re-run this cell.")

## <span style="color:#B5121B">4. Let the period float: a nonlinear fit</span>

Perhaps 12.42 h is the wrong period for this record. If $P$ is a free parameter the model is no longer linear in its parameters, so we need `scipy.optimize.curve_fit` (Levenberg–Marquardt, slide 19), which needs a starting guess `p0`.

The function `tide_model` below takes the parameters as `(c, a1, b1, a2, b2, ..., P)`, with harmonics $k\omega$ for $k = 1, 2, \dots$. Fit it with three harmonics (8 parameters), `sigma=np.full_like(level, sigma_real)` and `absolute_sigma=True`, starting from $P_0$ = 12, 13, 11 and 6 hours (amplitudes start at 0.5, 0.5 then zeros; $c$ at 2.8).

* Which starting values give the same answer? What do the others find, and how does their χ²/ν compare?
* For the best fit, how far is $P$ from 12.42 h, in units of its uncertainty? Repeat with `absolute_sigma=False`, which rescales the errors by $\sqrt{\chi^2/\nu}$. Does that change your conclusion?

In [ ]:
from scipy.optimize import curve_fit

def tide_model(t, *p):
    """c + sum_k [a_k cos(k w t) + b_k sin(k w t)], w = 2 pi / P; p = (c, a1, b1, ..., P)."""
    c, P = p[0], p[-1]
    w = 2 * np.pi / P
    out = np.full_like(t, c, dtype=float)
    n_harm = (len(p) - 2) // 2
    for k in range(1, n_harm + 1):
        out += p[2 * k - 1] * np.cos(k * w * t) + p[2 * k] * np.sin(k * w * t)
    return out

In [ ]:
# TODO: for P0 in [12, 13, 11, 6]:
#     p0 = [2.8, 0.5, 0.5, 0, 0, 0, 0, P0]
#     p, C = curve_fit(tide_model, t, level, p0=p0, sigma=np.full_like(level, sigma_real),
#                      absolute_sigma=True, maxfev=50000)
#     print P, its uncertainty, chi2/nu

## <span style="color:#B5121B">5. Why the starting guess matters: χ² against the period</span>

With the period fixed, the fit is linear and takes a fraction of a second. So we can map the whole χ² landscape that `curve_fit` was exploring: fit the three-harmonic model, `[P, P/2, P/3]`, at 2000 periods between 5 and 26 h, and record χ² each time.

1. Plot χ²/ν against $P$ (log $y$ axis) and mark where your four `curve_fit` runs ended. Why did they end up in different places?
2. Zoom in: scan ±0.0003 h around the best period with 601 points and plot $\chi^2 - \chi^2_\text{min}$. The 1σ range of $P$ is where this stays below 1 (Lecture 3, slide 24). Compare half its width with the uncertainty `curve_fit` gave.

In [ ]:
try:
    # TODO: P_grid = np.linspace(5, 26, 2000)
    #       chi2_grid = [linear_fit(t, level, sigma_real, [P, P / 2, P / 3])[2] for P in P_grid]
    # TODO: plot chi2/nu against P (ax.semilogy); mark the curve_fit results with ax.axvline
    # TODO: fine scan around the best P; plot chi2 - chi2.min(); find where it is below 1
    raise NotImplementedError
except NotImplementedError:
    print("Not implemented yet: complete the functions above, then re-run this cell.")

## <span style="color:#B5121B">6. What's missing? A second tide</span>

The Sun raises a tide too: the **S2** solar semi-diurnal tide, period exactly 12.00 h. M2 and S2 drift in and out of step every

$$\left(\frac{1}{12.00} - \frac{1}{12.42}\right)^{-1} \approx 355\ \text{h} \approx 14.8\ \text{days},$$

producing **spring tides** (in step, large range) and **neap tides** (out of step, small range). Over our 8 days, the sum looks like a single tide with a period between 12.00 and 12.42 h and a changing amplitude, which is what the free-period fit found.

With both periods fixed the model is linear again. Fit

`[M2, S2, M2/2, M2/3, S2/2, MS4]`, where `MS4 = 1 / (1/M2 + 1/S2)` is the compound tide from the two together,

and compare the rms residual and χ²/ν with the free-period fit of Section 4. Then print the correlation matrix of the four main M2 and S2 coefficients (columns 1–4), $\rho_{jk} = C_{jk}/\sqrt{C_{jj}C_{kk}}$. Why are some of them correlated, and what would reduce the correlation?

In [ ]:
S2 = 12.0
MS4 = 1 / (1 / M2 + 1 / S2)
periods_ms = [M2, S2, M2 / 2, M2 / 3, S2 / 2, MS4]
try:
    # TODO: fit with linear_fit; print rms residual and chi2/nu
    # TODO: print M2 and S2 amplitudes, sqrt(a^2 + b^2)
    # TODO: correlation matrix of theta[1:5]
    raise NotImplementedError
except NotImplementedError:
    print("Not implemented yet: complete the functions above, then re-run this cell.")

## <span style="color:#B5121B">7. Judge the final fit</span>

χ²/ν is still in the thousands. Following the lecture's checklist, look at the residuals before blaming the noise:

1. plot the data and the M2 + S2 model for the first 48 hours;
2. plot the residual against the water level (`plt.plot(level, level - model_ms, ".")`).

Where does the model fail? Look back at the Week 1 histogram. Why would it be wrong to "fix" χ²/ν by increasing σ until it equals 1 (slide 22)?

In [ ]:
try:
    # TODO: two panels: (1) data and model for t < 48 h; (2) residual against level
    raise NotImplementedError
except NotImplementedError:
    print("Not implemented yet: complete the functions above, then re-run this cell.")

## The verdict

In your logbook:

1. Why test on generated data first? What would you have missed without the pulls?
2. Which of your fits were linear and which nonlinear, and why does that matter in practice?
3. The free-period fit gave $P$ = 12.30 h with a tiny error bar. What was wrong with that answer, and how did the residuals tell you?
4. What would you need (data or model) to measure the M2 and S2 amplitudes properly?

## <span style="color:#FD9029">Extension (optional, unassessed): UK graduate earnings</span>

A different kind of data. `graduate_earnings.csv` (from the UK government's graduate labour-market statistics, used in PHYS465) gives median salaries from 2007 to 2020 with uncertainties, split by `gender` and `graduate_type`.

1. Select `gender == "all"` and `graduate_type == "graduate"`. Fit a straight line with `curve_fit`, using `sigma=income_err` and `absolute_sigma=True`. What is χ²/ν? Now plot the residuals. Do the two tell the same story?
2. Fit quadratic and cubic models (`np.polyfit` with `w=1/income_err` works). What happens to χ²/ν, and why is a very small χ²/ν suspicious (slide 23)?
3. Fit the straight line using `year` directly, then `year - 2007`. Compare the correlation between intercept and slope (slide 24).
4. These are *nominal* salaries. What else would you want to know before saying that graduate earnings rose?

In [ ]:
earnings = pd.read_csv(data_path("graduate_earnings.csv"))
# TODO

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>